# Fake Jobs – Exp 1: PyOD-Baselines
- iForest, LODA, ECOD, AutoEncoder (alle unsupervised, Originalverteilung)
- HPO über Val-Split (AP), Test unberührt; Logging nach MLflow
- AutoEncoder läuft auf GPU (`device="cuda"`)

In [ ]:
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

## Daten & Split
- Outlier = `fraudulent == 1`; 70/30 stratifiziert (seed 42)

In [ ]:
df = pd.read_csv("../../data/preprocessed/cleaned_fake_jobs.csv")
y = df["fraudulent"].values
X = df.drop(columns=["row_id", "fraudulent"]).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.3, stratify=y_train, random_state=42)
print("train", X_train.shape, "test", X_test.shape, "test outlier rate", round(y_test.mean(), 4))

## MLflow

In [ ]:
mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("fake_jobs_experiment_1")

## GridSearch (AP auf Val), Refit auf vollem Train, Test bewerten
- AutoEncoder: Originalverteilung wie die übrigen Baselines, GPU. ECOD: parameterfrei.

In [ ]:
models = {
    "iforest": (IForest, {"n_estimators": [100, 200], "max_features": [0.5, 1.0], "random_state": [42]}, False),
    "loda": (LODA, {"n_bins": [10, 20], "n_random_cuts": [100, 200]}, False),
    "ecod": (ECOD, {}, False),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [[64, 32], [32, 16]], "epoch_num": [20, 50], "random_state": [42], "device": ["cuda"]}, False),
}

for name, (Model, grid, inlier_only) in models.items():
    t0 = time.perf_counter()
    Xfit_tr = X_tr[y_tr == 0] if inlier_only else X_tr
    best_ap, best_params = -1.0, {}
    for params in (list(ParameterGrid(grid)) or [{}]):
        m = Model(**params)
        m.fit(Xfit_tr)
        ap = average_precision_score(y_val, m.decision_function(X_val))
        if ap > best_ap:
            best_ap, best_params = ap, params
    Xfit_full = X_train[y_train == 0] if inlier_only else X_train
    model = Model(**best_params)
    model.fit(Xfit_full)
    scores = model.decision_function(X_test)
    runtime = time.perf_counter() - t0
    ap = average_precision_score(y_test, scores)
    prec, rec, _ = precision_recall_curve(y_test, scores)
    auprc = auc(rec, prec)
    auroc = roc_auc_score(y_test, scores)
    with mlflow.start_run(run_name=name):
        mlflow.log_params(best_params)
        mlflow.log_metric("average_precision", ap)
        mlflow.log_metric("auprc", auprc)
        mlflow.log_metric("auc_roc", auroc)
        mlflow.log_metric("runtime_s", runtime)
    print(f"{name}: AP={ap:.4f} AUPRC={auprc:.4f} AUC={auroc:.4f} time={runtime:.1f}s params={best_params}")